In [1]:
"""
Calculate the average TCC within fire perimeters (or other boundary)
"""

import os, sys, time
import ee

# Custom functions
sys.path.append(os.path.join(os.getcwd(),'code/'))
from __functions import *

ee.Authenticate()
ee.Initialize(project='cfri-ee')

# local data directories
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Success !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Success !


In [2]:
tcc = ee.ImageCollection("USGS/NLCD_RELEASES/2023_REL/TCC/v2023-5")
print(tcc.first().bandNames().getInfo())

# Look at one image's properties (server-side)
print(tcc.first().toDictionary().keys().getInfo())

# See distinct values for likely year fields
for prop in ["startYear", "endYear", "year", "system:index"]:
    vals = tcc.aggregate_array(prop).distinct().sort()
    print(prop, vals.getInfo())

['Science_Percent_Tree_Canopy_Cover', 'Science_Percent_Tree_Canopy_Cover_Standard_Error', 'NLCD_Percent_Tree_Canopy_Cover', 'data_mask']
['endYear', 'startYear', 'study_area', 'version', 'year']
startYear [1985]
endYear [2023]
year [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
system:index ['TCC_v2023-5_AK_1985', 'TCC_v2023-5_AK_1986', 'TCC_v2023-5_AK_1987', 'TCC_v2023-5_AK_1988', 'TCC_v2023-5_AK_1989', 'TCC_v2023-5_AK_1990', 'TCC_v2023-5_AK_1991', 'TCC_v2023-5_AK_1992', 'TCC_v2023-5_AK_1993', 'TCC_v2023-5_AK_1994', 'TCC_v2023-5_AK_1995', 'TCC_v2023-5_AK_1996', 'TCC_v2023-5_AK_1997', 'TCC_v2023-5_AK_1998', 'TCC_v2023-5_AK_1999', 'TCC_v2023-5_AK_2000', 'TCC_v2023-5_AK_2001', 'TCC_v2023-5_AK_2002', 'TCC_v2023-5_AK_2003', 'TCC_v2023-5_AK_2004', 'TCC_v2023-5_AK_2005', 'TCC_v2023-5_AK_2006', 'TCC_v2023-5_

In [3]:
# --- Grab the NLCD TCC
tcc_nlcd = tcc.select('NLCD_Percent_Tree_Canopy_Cover')

# --- Select NLCD TCC band and filter to CONUS + year 2013
tcc_2013 = ee.Image(
    tcc
    .filter(ee.Filter.eq("study_area", "CONUS"))
    .filter(ee.Filter.eq("year", 2013))
    .first()
).select(['NLCD_Percent_Tree_Canopy_Cover'])
print(tcc_2013.bandNames().getInfo())

# --- Grab the fire data
perims = ee.FeatureCollection('projects/cfri-ee/assets/reshape/incidents_joined_EE')

# --- Run the reduction for ca. 2013 TCC
reduc = tcc_2013.reduceRegions(
    collection=perims,
    reducer=ee.Reducer.mean().setOutputs(['tcc']),
    scale=30, # resolution of the TCC
    tileScale=2
)

# Pull a few features into Python to inspect
sample = reduc.limit(10).getInfo()
props = [f['properties'] for f in sample['features']]
df = pd.DataFrame(props)
df

['NLCD_Percent_Tree_Canopy_Cover']


,START_YEAR,inci_id,tcc
0,2014,2014_569807_COLEMAN,2.469724
1,2014,2014_685752_MALT,1.077028
2,2014,2014_618898_DENIO BASIN,0.432266
3,2014,2014_520806_BRYANT,39.367632
4,2014,2014_991327_BONE CREEK BASIN,4.545846
5,2014,2014_977764_ONION MTN,76.354914
6,2014,2014_630309_MOCCASIN HILL,7.995755
7,2014,2014_993789_BLITZEN CROSSING,1.706318
8,2014,2014_875523_790 FIRE,55.991254
9,2014,2014_804581_FIFTEEN CENT,1.833027


In [4]:
# Remove geometry
def remove_geometry(ftr):
    return ftr.setGeometry(None)
reduc = reduc.map(remove_geometry)

def monitor_export(task, timeout=30):
    """ Monitors EE export task """
    while task.active():
        print('Waiting for export to finish..\n\tPatience young padawan.')
        time.sleep(timeout)  # Check every 30 seconds

    # Get the status of the task
    status = task.status()

    # Check if the task failed or succeeded
    if status['state'] == 'COMPLETED':
        print("Export completed successfully !!!!")
    elif status['state'] == 'FAILED':
        print(f"Export failed! Bummer. Reason: {status.get('error_message', 'Unknown error')}")
    else:
        print(f"Export ended with state: {status['state']}")

# export to drive
export_task = ee.batch.Export.table.toDrive(
    collection=reduc,
    description='incidents_2014to2023_tcc',
    fileNamePrefix='incidents_2014to2023_tcc',
    fileFormat='CSV',
    folder='ReSHAPE'
)
export_task.start() # Start the export task
print("Export to Earth Engine Asset started!")
# Monitor the task until it's finished
monitor_export(export_task, 30) # check every 30sec

Export to Earth Engine Asset started!
Waiting for export to finish..
	Patience young padawan.
Export completed successfully !!!!


In [5]:
# download the table from Google Drive ...

In [8]:
# --- Load the exported table
fp = os.path.join(projdir,'data/spatial/mod/ee/incidents_2014to2023_tcc.csv')
tcc = pd.read_csv(fp)

# --- Load the spatial data
fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters.gpkg')
incis = gpd.read_file(fp)

# --- Merge with the TCC summary
incis = incis.merge(tcc, left_on='INCIDENT_ID', right_on='inci_id')
incis['tcc'] = incis['tcc'].fillna(0)

# print some info
print(incis[['INCIDENT_ID','tcc']].head())
print(incis['tcc'].describe())
print(f"\nIncidents with >10% TCC: {len(incis[incis['tcc']>10])}")
print(f"Incidents with Timber fuel model: {len(incis[
    incis["FUEL_MODEL"].isin(['Timber (Grass and Understory)', 'Timber (Litter and Understory)'])
])}")

                     INCIDENT_ID  tcc
0      2021_12722173_PILOT POINT  0.0
1       2015_2896784_TWIN CREEKS  0.0
2    2022_14549956_CONTACT CREEK  0.0
3  2015_2815113_COPENHAGEN CREEK  0.0
4       2015_2716445_PAULS CREEK  0.0
count    6940.000000
mean       22.749407
std        26.738740
min         0.000000
25%         0.055085
50%        10.024563
75%        40.437139
max        94.149967
Name: tcc, dtype: float64

Incidents with >10% TCC: 3471
Incidents with Timber fuel model: 2765


In [9]:
incis['FUEL_MODEL'].unique()

array(['Short Grass (1 foot)', 'Heavy Logging Slash', 'Brush (2 feet)',
       'Timber (Grass and Understory)', 'Timber (Litter and Understory)',
       None, 'Medium Logging Slash', 'Tall Grass (2.5 feet)',
       'Hardwood Litter', 'Closed Timber Litter',
       'Dormant Brush, Hardwood Slash', 'Southern Rough',
       'Chaparral (6 feet)', 'Light Logging Slash'], dtype=object)

In [10]:
# --- Check the acre difference
pd.set_option("display.float_format", "{:,.4f}".format)
incis['acre_diff_abs'].describe()

count    6,940.0000
mean       489.5167
std      1,422.3244
min          0.0000
25%          3.0000
50%         63.0000
75%        304.0000
max     36,144.2949
Name: acre_diff_abs, dtype: float64

In [11]:
# --- Save the file out
out_fp = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis.to_file(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg
